In [1]:
# Match_Siblings_To_Soldiers_Partitions.py
# 10/16/2025

# Match Soldiers Using Siblings of Jewish Cemeteries Parents

# Rank_Dictionary_Lookup.py
# 10/10/2025
from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd


import json

import sys
import random


def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1

def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ','TEC5 ',\
                'PVT ','S Sgt ','Capt ','Pfc ','TSgt ','COL ','Tec5 ','TEC4 ','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant ','TEC 5 ','Maj ','GM3 ','TEC 4 ','Tec4 ','RM3 ','Corp ','T 5 ','MS ','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df 

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    processing_time_minutes=processing_time_seconds/60
    return f" {processing_time_minutes:.1f}"

def time_now():
    time_now=datetime.datetime.now()  
    return time_now

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df

def all_parts_in(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def expand_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            new_full_name=re.sub(pattern, rank, full_name.lower(), count=1)
            return new_full_name.title()
    return full_name  # return unchanged if no code found


def strip_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        if pd.isna(full_name):return None
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code              
        if re.search(pattern, full_name):
            new_full_name = re.sub(pattern, "", full_name).strip()   
            return  new_full_name
    return full_name  # return unchanged if no code found


def extract_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            # new_full_name = re.sub(pattern, "", full_name.lower()).strip()           
            return  rank
    return ''  # return unchanged if no code found

def extract_rank_multiple(full_name,n):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if len(re.findall(pattern, full_name.lower()))>=n:
            # new_full_name = re.sub(pattern, "", full_name.lower()).strip()           
            return  rank
    return ''  # return unchanged if no code found

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

def get_tic():
    tic=datetime.datetime.now()
    return tic

blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

# print("***Jewish_Cemetery_Siblings_Soldiers_Match--- Started *****")
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileABMCInput="ABMC_FAG_Soldiers_Merge_Master"
fileSoDInput="star_david_20251013_df"
fileRankInput="Rank_Dictionary"

fileABMCIn=DirMain+"0-Master Worksheets\\"+fileABMCInput+".csv"
fileSoDIn=DirMain+"0-Master Worksheets\\"+fileSoDInput+".csv"
fileRankIn=DirMain+"0-Master Worksheets\\"+fileRankInput+".csv"
tic_start=get_tic()
tic=get_tic()

soldiers_df=pd.read_csv(fileABMCIn,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
star_david_df=pd.read_csv(fileSoDIn)
rank_df=pd.read_csv(fileRankIn)


rank_lookup = rank_df.set_index("abbreviated_rank")["rank"].to_dict()
soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].str.lower()
# print('Lower:',soldiers_df["soldiers_full_name"])


soldiers_df=soldiers_df.copy()
print("$$$$$ Starting soldiers_full_name Strip Rank $$$") 
soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].apply(strip_rank)
process_time= time_lapse(tic)
process_time_cum= time_lapse(tic_start)
tic=get_tic()
print(f"$$$$$ soldiers_full_name Strip Rank Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n')
# print('Strip:',soldiers_df["soldiers_full_name"])
# input('')


##################################### Partitioned Inputs ###############################################################################################
group_A=['Jewish_Graves','Jewish_Cemeteries_All_Graves_','Reverse_Parents',0,27]

partition_df=pd.DataFrame([group_A])
partition_df_list=['directory','format','label','range_start','range_end']
partition_df.columns=[partition_df_list]

parents_df=pd.DataFrame([])
for index, row in partition_df.iterrows():
    subdirectory=row['directory']
    file_format=row['format']
    label=row['label']
    partition_start=row['range_start']
    partition_end=row['range_end']
    fileOutParents=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\Parents_Soldier_Reverse_Match_"+label+"_"+timestamp+".csv"
    start=partition_start
    end=partition_end
    parents_df=pd.DataFrame([])
    missing_cemetery_id_df=pd.DataFrame([])

    for file_number in range(start,end):
        file_number_str='0'+str(file_number)
        file_number_str=file_number_str[-2:]
        fileInParents=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Inputs\\"+subdirectory+"\\"+file_format+file_number_str+".csv"
        parents_partitions_df=pd.read_csv(fileInParents,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
    
        parents_partitions_df = parents_partitions_df.dropna(subset=['Siblings'])                      
        parents_df=pd.concat([parents_df,parents_partitions_df])
        parents_df=drop_sets_dataframe_duplicates(parents_df)
        parents_df.reset_index(drop=True, inplace = True)
        print(f" {blank_line}", end='\r')
        print(f"Group: {label} • Uploading File {file_format+file_number_str} •  Number of Partitions: {end} • Record Count: {len(parents_partitions_df):,}/ {len(parents_df):,}", end='\r') 
        parents_df["Siblings"]= parents_df["Siblings"].astype(str).replace('\.', '', regex=True).str.strip()
    print('\n')
    print(f"Group: {label} • Uploaded All {end} Partitions • Record Count: {len(parents_df):,}") 
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Upload Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' ) 
    parents_df["Siblings_rank1"] = parents_df["Siblings"].apply(lambda x: extract_rank_multiple(x, 1))
    parents_df["Siblings_rank2"] = parents_df["Siblings"].apply(lambda x: extract_rank_multiple(x, 2))
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Extracting Siblings's Rank Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )    
    # print('Rank1:',parents_df["Siblings"][parents_df["Siblings_rank1"]!=''],'\n\n')
    # print('Rank2:',parents_df["Siblings"][parents_df["Siblings_rank2"]!=''])

    print(f" Group Type Uploaded: {label} • Records: {len(parents_df):,} ")        
    file_parents_df=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\parents_df_"+label+"_"+timestamp+".csv"
    parents_df.to_csv(file_parents_df,mode="w",index=False,header=True)
    print('$$$$$ Partition Downloads Complete $$$\n')

    parents_short_df_list=['lastName','firstName','middleName','fullName','cemeteryName','plot','cemeteryId','intermentHasPhoto','birthYear','birthStateName'\
     ,'birthCountryName','Spouses','Siblings','Siblings_rank1','Siblings_rank2']
    
    soldiers_short_df_list=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','missing_status'\
      ,'birthYear','entry_state','soldiers_cemetery','plot','birthCity','birthState','birthCountry','Parents']
    
    parents_short_df=parents_df[parents_short_df_list]
    soldiers_short_df=soldiers_df[soldiers_short_df_list]
    
    matching_parents_soldiers_df=pd.merge(
        soldiers_short_df,
        parents_short_df,
        left_on=['soldiers_last_name'],
        right_on=['lastName'],    
        how='inner')
    
    matching_parents_soldiers_df_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Parents','lastName','fullName'\
                ,'Siblings','Siblings_rank1','Siblings_rank2','cemeteryName','plot_y','cemeteryId','intermentHasPhoto','Spouses','birthStateName'\
                ,'birthCountryName','missing_status','birthYear_y','soldiers_cemetery'\
                  ,'plot_x','entry_state','birthState','birthCountry','birthYear_x']

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Match parent to Soldier - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )   
    
    print(" **** Starting Needle in Haystack: Soldier's Name: ")
    needle='soldiers_full_name'
    haystack='Siblings'
    
    dataframe_df=matching_parents_soldiers_df[matching_parents_soldiers_df_list]

# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$    
    dataframe_df=dataframe_df.copy()
    dataframe_df['Siblings']=dataframe_df['Siblings'].str.lower()
    dataframe_df['soldiers_full_name']=dataframe_df['soldiers_full_name'].str.lower()
    dataframe_df["soldiers_full_name"]=dataframe_df["soldiers_full_name"].apply(strip_rank)

    matching_parents_soldiers_short_df, mask_name=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    matching_parents_soldiers_short_df=matching_parents_soldiers_short_df[matching_parents_soldiers_short_df['Siblings_match']==True]
    matching_parents_soldiers_short_df.reset_index(drop=True, inplace = True)

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Needle in Haystack: Soldier's Name - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )       
    
    # First rank Match 
    print(' **** Starting Needle in Haystack: Rank 1 **** ')
    dataframe_df=matching_parents_soldiers_short_df
    needle='military_rank'
    haystack='Siblings_rank1'
    
    matching_parents_soldiers_short_df, mask_rank1=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    
    # Second rank Match 
    print(' **** Starting Needle in Haystack: Rank 2 **** ')
    dataframe_df=matching_parents_soldiers_short_df
    needle='military_rank'
    haystack='Siblings_rank2'
    
    matching_parents_soldiers_short_df, mask_rank2=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Needle in Haystack: Siblingss Rank 1 and 2 - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )       
    
    file_parents_soldier_match=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\matching_parents_soldiers_df_"+timestamp+".csv"
    matching_parents_soldiers_short_df.to_csv(file_parents_soldier_match,mode="w",index=False,header=True)
    print('$$$$$ matching_parents_soldiers_df Downloads Complete $$$\n')
    
    print(' **** Starting Star_of_David Association ')
    parents_soldiers_match_sod_df=pd.merge(
        matching_parents_soldiers_short_df,
        star_david_df,
        left_on=[ 'name_index'],
        right_on=['sod_Name_Index'],
        how='left')
    
    parents_soldiers_match_sod_df['Star_of_David'] = parents_soldiers_match_sod_df['sod_Name_Index'].fillna('False')
    parents_soldiers_match_sod_df['Star_of_David']=parents_soldiers_match_sod_df['Star_of_David']!='False'
    parents_soldiers_match_sod_df.drop_duplicates(inplace=True)
    
    reorder_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Siblings', 'Siblings_rank1'\
                  ,'Siblings_rank2', 'Siblings_match','Siblings_rank1_match','Siblings_rank2_match','Star_of_David','Parents'\
                  ,'fullName','Spouses','cemeteryName','plot_y','cemeteryId','birthStateName','birthCountryName','birthYear_y'\
                  ,'missing_status','entry_state','birthState','birthCountry','soldiers_cemetery','birthYear_x','plot_x']
    
    rename_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Siblings', 'Siblings_rank1'\
                  ,'Siblings_rank2', 'Siblings_match','Siblings_rank1_match','Siblings_rank2_match','Star_of_David'\
                 ,'Parents','parent_full_name','spouse','parents_cemetery','plot_parents','cemetery_id'\
                ,'parent_birth_state','parent_birth_country','parent_birth_year'\
                 ,'missing_status','entry_state','birth_state','birth_country','soldiers_cemetery','soldiers_birth_year','soldiers_plot']
    
    parents_soldiers_match_sod_df=parents_soldiers_match_sod_df[reorder_list]
    parents_soldiers_match_sod_df.columns=rename_list
    parents_soldiers_match_sod_df['soldiers_full_name']=parents_soldiers_match_sod_df['soldiers_full_name'].str.title()
    parents_soldiers_match_sod_df['Siblings']=parents_soldiers_match_sod_df['Siblings'].str.title()
    
    print(' **** Start Saving - parents_soldiers_match_df ')
    file_parents_df=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\parents_soldiers_match_sod_df_"+label+"_"+timestamp+".csv"
    parents_soldiers_match_sod_df.to_csv(file_parents_df,mode="w",index=False,header=True)
    print('$$$$$ Parent_Soldiers Matching Complete $$$ - Size: ',len(parents_soldiers_match_df))

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    print(f"$$$$$ Parent-Soldiers SOD - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )  


    print(f"$$$$$ Parent-Soldiers Classification  - Complete $$$ - Total Processing Time: {process_time_cum/60:4.1f} Minutes",'\n' )  
    


$$$$$ Starting soldiers_full_name Strip Rank $$$
$$$$$ soldiers_full_name Strip Rank Complete $$$ - Process Time:  0.4/ 0.4 Minutes 

Group: Reverse_Parents • Uploading File Jewish_Cemeteries_All_Graves_26 •  Number of Partitions: 27 • Record Count: 8,401/ 303,162                                                          

Group: Reverse_Parents • Uploaded All 27 Partitions • Record Count: 303,162
$$$$$ Upload Complete $$$ - Process Time:  1.3/ 1.7 Minutes 

$$$$$ Extracting Siblings's Rank Complete $$$ - Process Time:  5.1/ 6.7 Minutes 

 Group Type Uploaded: Reverse_Parents • Records: 303,162 
$$$$$ Partition Downloads Complete $$$

$$$$$ Match parent to Soldier - Complete $$$ - Process Time:  0.1/ 6.8 Minutes 

 **** Starting Needle in Haystack: Soldier's Name: 
$$$$$ Needle in Haystack: Soldier's Name - Complete $$$ - Process Time: 32.1/38.9 Minutes 

 **** Starting Needle in Haystack: Rank 1 **** 
 **** Starting Needle in Haystack: Rank 2 **** 
$$$$$ Needle in Haystack: Siblingss R

NameError: name 'parents_soldiers_match_df' is not defined